# 01 — Data Loading & Enrichment Pipeline

Load raw SIH and CNES parquet files, filter to kidney stone (N20) admissions in São Paulo,
enrich with facility characteristics (equipment, beds, staff, governance), compute hospital
classification tags, and save clean datasets for all downstream notebooks.

**Outputs:**
- `kidney_sih.parquet` — N20 admissions with parsed dates and types
- `hospital_tags.parquet` — one row per hospital with classification tags
- `cnes_enriched.parquet` — facility features (equipment, beds, staff, governance)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

from shared import (
    OUTPUT_DIR, DATA_DIR, RAW_DATA_DIR, RAW_SIH_DIR, RAW_CNES_DIR,
    SIH_COLS, CATEGORY_MAP, FACILITY_TYPE_MAP, classify_legal_nature,
)

warnings.filterwarnings("ignore", category=FutureWarning)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data directory: {RAW_DATA_DIR.resolve()}")
print(f"Output directory:   {DATA_DIR.resolve()}")

Raw data directory: /Users/gsampaio/redhat/customers/prodesp/health-sus-agent/data
Output directory:   /Users/gsampaio/redhat/customers/prodesp/health-sus-agent/experiments/kidney/outputs/data


## 1. Load SIH Data

SIH AIH Reduzida files are partitioned parquets: `data/sih/RDSPYYMM.parquet/`.
Each directory contains hospital admission records for one month.

In [2]:
sih_files = sorted(RAW_SIH_DIR.glob("RDSP*.parquet"))
print(f"Found {len(sih_files)} SIH parquet directories")
print(f"Date range: {sih_files[0].name} to {sih_files[-1].name}")

frames = []
for f in sih_files:
    df = pd.read_parquet(f)
    available = [c for c in SIH_COLS if c in df.columns]
    frames.append(df[available])

sih = pd.concat(frames, ignore_index=True)
print(f"Total SIH records: {len(sih):,}")

Found 120 SIH parquet directories
Date range: RDSP1601.parquet to RDSP2512.parquet


Total SIH records: 25,787,169


## 2. Filter to Kidney Stone (N20)

In [3]:
sih["DIAG_PRINC"] = sih["DIAG_PRINC"].astype(str).str.strip()
kidney = sih[sih["DIAG_PRINC"].str.startswith("N20")].copy()
del sih  # free memory

print(f"Kidney stone (N20) admissions: {len(kidney):,}")
print(f"\nSub-diagnoses:")
print(kidney["DIAG_PRINC"].value_counts())

Kidney stone (N20) admissions: 206,500

Sub-diagnoses:
DIAG_PRINC
N201    80523
N200    72093
N202    39539
N209    12427
N20      1918
Name: count, dtype: int64


## 3. Parse Dates & Types

In [4]:
kidney["DT_INTER"] = pd.to_datetime(kidney["DT_INTER"], format="%Y%m%d", errors="coerce")
kidney["year"] = kidney["DT_INTER"].dt.year
kidney["month"] = kidney["DT_INTER"].dt.month

kidney["DIAS_PERM"] = pd.to_numeric(kidney["DIAS_PERM"], errors="coerce").fillna(0).astype(int)
kidney["VAL_TOT"] = pd.to_numeric(kidney["VAL_TOT"], errors="coerce").fillna(0)
kidney["IDADE"] = pd.to_numeric(kidney["IDADE"], errors="coerce").fillna(0).astype(int)
kidney["MORTE"] = pd.to_numeric(kidney["MORTE"], errors="coerce").fillna(0).astype(int)

print(f"Year range: {kidney['year'].min()} to {kidney['year'].max()}")
print(f"Missing DT_INTER: {kidney['DT_INTER'].isna().sum()}")
print(f"\nBasic stats:")
kidney[["DIAS_PERM", "VAL_TOT", "IDADE"]].describe()

Year range: 2015 to 2025
Missing DT_INTER: 0

Basic stats:


,DIAS_PERM,VAL_TOT,IDADE
count,206500.000000,206500.000000,206500.000000
mean,2.457458,909.555233,46.316596
std,3.140623,980.671059,15.265192
min,0.000000,0.000000,0.000000
25%,1.000000,414.680000,35.000000
50%,2.000000,766.110000,46.000000
75%,3.000000,1130.070000,57.000000
max,97.000000,69163.680000,98.000000


## 4. Load CNES Establishment Data (ST)

Use the most recent CNES snapshot for facility characteristics. Join on `CNES` facility ID.

In [5]:
cnes_files = sorted(RAW_CNES_DIR.glob("STSP*.parquet"))
print(f"Found {len(cnes_files)} CNES establishment files")

cnes_st = pd.read_parquet(cnes_files[-1])
cnes_st["CNES"] = cnes_st["CNES"].astype(str).str.strip()
print(f"CNES records (latest snapshot): {len(cnes_st):,}")

# Hospitals that treated kidney stone patients
kidney_hospitals = kidney["CNES"].astype(str).str.strip().unique()
print(f"Unique hospitals treating N20: {len(kidney_hospitals)}")

cnes_cols = ["CNES", "CODUFMUN", "TP_UNID", "NAT_JUR", "VINC_SUS",
             "AV_ACRED", "AV_PNASS", "COLETRES"]
# Add committee columns if present
comm_cols = [f"COMISS{i:02d}" for i in range(1, 13)]
cnes_cols += [c for c in comm_cols if c in cnes_st.columns]

available_cols = [c for c in cnes_cols if c in cnes_st.columns]
cnes_base = cnes_st[available_cols].drop_duplicates(subset=["CNES"])
print(f"CNES base features: {len(available_cols)} columns")

Found 120 CNES establishment files


CNES records (latest snapshot): 109,849
Unique hospitals treating N20: 510
CNES base features: 20 columns


## 5. Load CNES Equipment Data (EQ)

Equipment inventory: CT scanners, lithotripters, endoscopes.

In [6]:
eq_files = sorted(RAW_CNES_DIR.glob("EQSP*.parquet"))
print(f"Found {len(eq_files)} CNES equipment files")

if eq_files:
    eq = pd.read_parquet(eq_files[-1])
    eq["CNES"] = eq["CNES"].astype(str).str.strip()

    # Aggregate equipment per facility
    eq_agg = eq.groupby("CNES").agg(
        total_equipment=pd.NamedAgg(column="CNES", aggfunc="count"),
    ).reset_index()

    # Count specific equipment types if CODEQUIP exists
    if "CODEQUIP" in eq.columns:
        eq["CODEQUIP"] = eq["CODEQUIP"].astype(str).str.strip()
        # CT scanners (codes starting with 01 in EQUIP type)
        ct_mask = eq["CODEQUIP"].str.startswith("01")
        ct_counts = eq[ct_mask].groupby("CNES").size().rename("ct_scanners")
        eq_agg = eq_agg.merge(ct_counts, on="CNES", how="left")
        eq_agg["ct_scanners"] = eq_agg["ct_scanners"].fillna(0).astype(int)

    # Equipment utilization if QT_USO exists
    if "QT_USO" in eq.columns:
        eq["QT_USO"] = pd.to_numeric(eq["QT_USO"], errors="coerce").fillna(0)
        eq["QT_EXIST"] = pd.to_numeric(eq.get("QT_EXIST", 0), errors="coerce").fillna(0)
        uso = eq.groupby("CNES").agg(
            equip_in_use=pd.NamedAgg(column="QT_USO", aggfunc="sum"),
            equip_existing=pd.NamedAgg(column="QT_EXIST", aggfunc="sum"),
        ).reset_index()
        uso["equip_utilization"] = np.where(
            uso["equip_existing"] > 0,
            uso["equip_in_use"] / uso["equip_existing"],
            0
        )
        eq_agg = eq_agg.merge(uso, on="CNES", how="left")

    print(f"Equipment features computed for {len(eq_agg)} facilities")
else:
    print("WARNING: No CNES EQ files found. Equipment features unavailable.")
    eq_agg = pd.DataFrame(columns=["CNES", "total_equipment"])

Found 24 CNES equipment files


Equipment features computed for 50460 facilities


## 6. Load CNES Bed Data (LT)

Bed types: surgical, ICU, clinical beds. Capacity indicator.

In [7]:
lt_files = sorted(RAW_CNES_DIR.glob("LTSP*.parquet"))
print(f"Found {len(lt_files)} CNES bed files")

if lt_files:
    lt = pd.read_parquet(lt_files[-1])
    lt["CNES"] = lt["CNES"].astype(str).str.strip()

    for col in ["QT_EXIST", "QT_SUS", "QT_CONTR"]:
        if col in lt.columns:
            lt[col] = pd.to_numeric(lt[col], errors="coerce").fillna(0)

    lt_agg = lt.groupby("CNES").agg(
        total_beds=pd.NamedAgg(column="QT_EXIST", aggfunc="sum"),
        total_beds_sus=pd.NamedAgg(column="QT_SUS", aggfunc="sum"),
    ).reset_index()

    # Bed type breakdown if TP_LEITO exists
    if "TP_LEITO" in lt.columns:
        lt["TP_LEITO"] = lt["TP_LEITO"].astype(str).str.strip()
        # Surgical beds = type 01, Clinical = 03, ICU = 74-76
        for bed_type, bed_name in [("01", "surgical_beds"), ("03", "clinical_beds")]:
            mask = lt["TP_LEITO"] == bed_type
            bed_counts = lt[mask].groupby("CNES")["QT_EXIST"].sum().rename(bed_name)
            lt_agg = lt_agg.merge(bed_counts, on="CNES", how="left")
            lt_agg[bed_name] = lt_agg[bed_name].fillna(0).astype(int)

        icu_mask = lt["TP_LEITO"].isin(["74", "75", "76"])
        icu_counts = lt[icu_mask].groupby("CNES")["QT_EXIST"].sum().rename("icu_beds")
        lt_agg = lt_agg.merge(icu_counts, on="CNES", how="left")
        lt_agg["icu_beds"] = lt_agg["icu_beds"].fillna(0).astype(int)

    lt_agg["sus_bed_fraction"] = np.where(
        lt_agg["total_beds"] > 0,
        lt_agg["total_beds_sus"] / lt_agg["total_beds"],
        0
    )

    print(f"Bed features computed for {len(lt_agg)} facilities")
else:
    print("WARNING: No CNES LT files found. Bed features unavailable.")
    lt_agg = pd.DataFrame(columns=["CNES", "total_beds"])

Found 24 CNES bed files
Bed features computed for 1462 facilities


## 7. Load CNES Professional Staff Data (PF)

Staffing ratios: doctors per bed, specialists.

In [8]:
pf_files = sorted(RAW_CNES_DIR.glob("PFSP*.parquet"))
print(f"Found {len(pf_files)} CNES professional files")

if pf_files:
    pf = pd.read_parquet(pf_files[-1])
    pf["CNES"] = pf["CNES"].astype(str).str.strip()

    pf_agg = pf.groupby("CNES").agg(
        total_professionals=pd.NamedAgg(column="CNES", aggfunc="count"),
    ).reset_index()

    # CBO (occupation) based breakdown if available
    if "CBO" in pf.columns:
        pf["CBO"] = pf["CBO"].astype(str).str.strip()
        # Doctors: CBO starting with 2251 (médicos)
        doc_mask = pf["CBO"].str.startswith("2251")
        doc_counts = pf[doc_mask].groupby("CNES").size().rename("doctors")
        pf_agg = pf_agg.merge(doc_counts, on="CNES", how="left")
        pf_agg["doctors"] = pf_agg["doctors"].fillna(0).astype(int)

        # Nurses: CBO 2235
        nurse_mask = pf["CBO"].str.startswith("2235")
        nurse_counts = pf[nurse_mask].groupby("CNES").size().rename("nurses")
        pf_agg = pf_agg.merge(nurse_counts, on="CNES", how="left")
        pf_agg["nurses"] = pf_agg["nurses"].fillna(0).astype(int)

        # Surgeons: CBO 225250 (cirurgião geral)
        surgeon_mask = pf["CBO"].str.startswith("22525")
        surgeon_counts = pf[surgeon_mask].groupby("CNES").size().rename("surgeons")
        pf_agg = pf_agg.merge(surgeon_counts, on="CNES", how="left")
        pf_agg["surgeons"] = pf_agg["surgeons"].fillna(0).astype(int)

    print(f"Staff features computed for {len(pf_agg)} facilities")
else:
    print("WARNING: No CNES PF files found. Staff features unavailable.")
    pf_agg = pd.DataFrame(columns=["CNES", "total_professionals"])

Found 24 CNES professional files
Staff features computed for 3512 facilities


## 8. Build Enriched CNES Dataset

Merge establishment, equipment, bed, and staff features into one facility-level dataset.

In [9]:
cnes_enriched = cnes_base.copy()

# Facility type classification
if "TP_UNID" in cnes_enriched.columns:
    cnes_enriched["facility_type"] = (
        cnes_enriched["TP_UNID"].astype(str).str.strip().map(FACILITY_TYPE_MAP).fillna("other")
    )

# Legal nature classification
if "NAT_JUR" in cnes_enriched.columns:
    cnes_enriched["legal_nature"] = cnes_enriched["NAT_JUR"].apply(classify_legal_nature)

# Governance score: count of active committees
comm_cols_available = [c for c in cnes_enriched.columns if c.startswith("COMISS")]
if comm_cols_available:
    for c in comm_cols_available:
        cnes_enriched[c] = pd.to_numeric(cnes_enriched[c], errors="coerce").fillna(0)
    cnes_enriched["active_committees"] = cnes_enriched[comm_cols_available].sum(axis=1).astype(int)

# Merge equipment features
if not eq_agg.empty:
    cnes_enriched = cnes_enriched.merge(eq_agg, on="CNES", how="left")

# Merge bed features
if not lt_agg.empty:
    cnes_enriched = cnes_enriched.merge(lt_agg, on="CNES", how="left")

# Merge staff features
if not pf_agg.empty:
    cnes_enriched = cnes_enriched.merge(pf_agg, on="CNES", how="left")

# Compute derived ratios
if "total_beds" in cnes_enriched.columns and "doctors" in cnes_enriched.columns:
    cnes_enriched["doctors_per_bed"] = np.where(
        cnes_enriched["total_beds"] > 0,
        cnes_enriched["doctors"] / cnes_enriched["total_beds"],
        0
    )
if "total_beds" in cnes_enriched.columns and "nurses" in cnes_enriched.columns:
    cnes_enriched["nurses_per_bed"] = np.where(
        cnes_enriched["total_beds"] > 0,
        cnes_enriched["nurses"] / cnes_enriched["total_beds"],
        0
    )

# Keep only hospitals that treated N20 patients
cnes_kidney = cnes_enriched[cnes_enriched["CNES"].isin(kidney_hospitals)].copy()
print(f"Enriched CNES: {len(cnes_enriched):,} total facilities")
print(f"Enriched CNES (kidney hospitals only): {len(cnes_kidney):,}")
print(f"Columns: {list(cnes_enriched.columns)}")

Enriched CNES: 109,849 total facilities
Enriched CNES (kidney hospitals only): 496
Columns: ['CNES', 'CODUFMUN', 'TP_UNID', 'NAT_JUR', 'VINC_SUS', 'AV_ACRED', 'AV_PNASS', 'COLETRES', 'COMISS01', 'COMISS02', 'COMISS03', 'COMISS04', 'COMISS05', 'COMISS06', 'COMISS07', 'COMISS08', 'COMISS09', 'COMISS10', 'COMISS11', 'COMISS12', 'facility_type', 'legal_nature', 'active_committees', 'total_equipment', 'ct_scanners', 'equip_in_use', 'equip_existing', 'equip_utilization', 'total_beds', 'total_beds_sus', 'surgical_beds', 'clinical_beds', 'icu_beds', 'sus_bed_fraction', 'total_professionals', 'doctors', 'nurses', 'surgeons', 'doctors_per_bed', 'nurses_per_bed']


## 9. Build Hospital Classification Tags

Every hospital gets tagged before any comparison or ranking:
1. **Facility type** — from CNES `TP_UNID`
2. **Admission profile** — elective-dominant, emergency-dominant, or mixed
3. **Case-mix profile** — surgical center, diagnostic-heavy, etc.
4. **Comparability group** — concatenation of the three tags above

In [10]:
# Compute hospital-level stats from SIH
kidney["PROC_REA"] = kidney["PROC_REA"].astype(str).str.strip()
kidney["proc_category"] = kidney["PROC_REA"].map(CATEGORY_MAP).fillna("OTHER")
kidney["CNES"] = kidney["CNES"].astype(str).str.strip()
kidney["CAR_INT"] = kidney["CAR_INT"].astype(str).str.strip()

hosp_stats = kidney.groupby("CNES").agg(
    n_admissions=pd.NamedAgg(column="CNES", aggfunc="count"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
    median_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="median"),
    total_bed_days=pd.NamedAgg(column="DIAS_PERM", aggfunc="sum"),
    total_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="sum"),
    mortality_rate=pd.NamedAgg(column="MORTE", aggfunc="mean"),
    n_deaths=pd.NamedAgg(column="MORTE", aggfunc="sum"),
).reset_index()

# Admission profile
adm_profile = kidney.groupby("CNES")["CAR_INT"].apply(
    lambda x: (x == "01").mean()
).rename("elective_rate").reset_index()

hosp_stats = hosp_stats.merge(adm_profile, on="CNES", how="left")
hosp_stats["emergency_rate"] = 1 - hosp_stats["elective_rate"]

hosp_stats["admission_profile"] = np.where(
    hosp_stats["elective_rate"] > 0.6, "elective_dominant",
    np.where(hosp_stats["emergency_rate"] > 0.6, "emergency_dominant", "mixed")
)

# Case-mix profile
proc_mix = kidney.groupby(["CNES", "proc_category"]).size().unstack(fill_value=0)
proc_mix_pct = proc_mix.div(proc_mix.sum(axis=1), axis=0)

def classify_case_mix(row):
    if row.get("SURGICAL", 0) > 0.5:
        return "surgical_center"
    if row.get("DIAGNOSTIC", 0) > 0.4:
        return "diagnostic_heavy"
    if row.get("CLINICAL_MGMT", 0) > 0.3:
        return "clinical_mgmt_focused"
    return "mixed_procedures"

proc_mix_pct["case_mix_profile"] = proc_mix_pct.apply(classify_case_mix, axis=1)
hosp_stats = hosp_stats.merge(
    proc_mix_pct[["case_mix_profile"]].reset_index(), on="CNES", how="left"
)

# Merge facility type from CNES
if "facility_type" in cnes_enriched.columns:
    hosp_stats = hosp_stats.merge(
        cnes_enriched[["CNES", "facility_type"]].drop_duplicates(subset=["CNES"]),
        on="CNES", how="left"
    )
    hosp_stats["facility_type"] = hosp_stats["facility_type"].fillna("unknown")
else:
    hosp_stats["facility_type"] = "unknown"

# Comparability group
hosp_stats["comparability_group"] = (
    hosp_stats["facility_type"] + "__" +
    hosp_stats["admission_profile"] + "__" +
    hosp_stats["case_mix_profile"].fillna("mixed_procedures")
)

hospital_tags = hosp_stats.copy()
print(f"Hospital tags computed for {len(hospital_tags)} hospitals")
print(f"\nComparability group distribution:")
print(hospital_tags["comparability_group"].value_counts().head(10))

Hospital tags computed for 510 hospitals

Comparability group distribution:
comparability_group
hospital_geral__emergency_dominant__diagnostic_heavy            244
hospital_geral__emergency_dominant__mixed_procedures             47
hospital_geral__mixed__mixed_procedures                          27
hospital_geral__elective_dominant__surgical_center               23
hospital_geral__emergency_dominant__surgical_center              21
hospital_geral__elective_dominant__mixed_procedures              18
pronto_socorro__emergency_dominant__diagnostic_heavy             12
hospital_especializado__emergency_dominant__diagnostic_heavy     12
unknown__emergency_dominant__diagnostic_heavy                    11
hospital_geral__emergency_dominant__clinical_mgmt_focused        11
Name: count, dtype: int64


## 10. Save Outputs

In [11]:
# Save kidney admissions
kidney.to_parquet(DATA_DIR / "kidney_sih.parquet", index=False)
print(f"Saved kidney_sih.parquet: {len(kidney):,} rows, {len(kidney.columns)} cols")

# Save hospital tags
hospital_tags.to_parquet(DATA_DIR / "hospital_tags.parquet", index=False)
print(f"Saved hospital_tags.parquet: {len(hospital_tags):,} rows")

# Save enriched CNES (all facilities, not just kidney)
cnes_enriched.to_parquet(DATA_DIR / "cnes_enriched.parquet", index=False)
print(f"Saved cnes_enriched.parquet: {len(cnes_enriched):,} rows")

# Save raw CNES for kidney hospitals
cnes_kidney.to_parquet(DATA_DIR / "kidney_cnes.parquet", index=False)
print(f"Saved kidney_cnes.parquet: {len(cnes_kidney):,} rows")

print("\nData loading complete. Proceed to 02_general_overview.ipynb.")

Saved kidney_sih.parquet: 206,500 rows, 24 cols
Saved hospital_tags.parquet: 510 rows


Saved cnes_enriched.parquet: 109,849 rows
Saved kidney_cnes.parquet: 496 rows

Data loading complete. Proceed to 02_general_overview.ipynb.
